# CUPED and pre-registered segments

Two analyses, both locked in `docs/experiment_design.md` (commit 5494da5) before this notebook was run.

- **CUPED** on history, for conversion and spend. The decision stays on the unadjusted readout in `02_primary_readout.ipynb`. CUPED is reported beside it as an efficiency result.
- **Segments**, 22 conversion tests (Mens vs No E-Mail and Womens vs No E-Mail in each of 11 levels), BH corrected, with FCR intervals matched to BH and a Newcombe interval beside each Wald interval. The targeted rollout tier is not triggered (both emails are significant overall), so segment results are descriptive and do not change who receives an email. A BH-significant negative lift is flagged as a risk.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import confint_proportions_2indep
from load import load_hillstrom

df = load_hillstrom()
TABLES = Path.cwd().parent / "results" / "tables"

ALPHA = 0.05
N_COMPARISONS = 3
ALPHA_ADJ = ALPHA / N_COMPARISONS
comparisons = [("Mens E-Mail", "No E-Mail"), ("Womens E-Mail", "No E-Mail"), ("Mens E-Mail", "Womens E-Mail")]
print(f"rows: {len(df)}")

rows: 64000


# Part A. CUPED

## A1. The adjustment

For each customer, with outcome Y and pre-period covariate X = history,

Y_adj = Y - theta (X - mean(X)), where theta = cov(Y, X) / var(X)

theta is estimated once, pooled over all 64,000 customers, so every arm is adjusted the same way (Deng et al. 2013). history was measured before the send, so the email cannot have changed it, and subtracting it does not bias the lift. It only removes the part of Y that history already predicts.

Shown below for each metric, cov(Y, X), var(X), theta, and a cross-check that theta equals the slope of a least squares line of Y on X.

In [2]:
x = df["history"].to_numpy(dtype=float)
x_mean = x.mean()
theta = {}
for metric in ["conversion", "spend"]:
    y = df[metric].to_numpy(dtype=float)
    cov_yx = np.cov(y, x, ddof=1)[0, 1]
    var_x = x.var(ddof=1)
    theta[metric] = cov_yx / var_x
    slope = np.polyfit(x, y, 1)[0]
    df[f"{metric}_cuped"] = y - theta[metric] * (x - x_mean)
    print(f"{metric}: cov(Y, X) {cov_yx:.6f}, var(X) {var_x:.4f}, theta {theta[metric]:.8e}, least squares slope {slope:.8e}")
    print(f"  overall mean raw {y.mean():.6f}, adjusted {df[f'{metric}_cuped'].mean():.6f}")

conversion: cov(Y, X) 0.712579, var(X) 65617.2326, theta 1.08596386e-05, least squares slope 1.08596386e-05
  overall mean raw 0.009031, adjusted 0.009031
spend: cov(Y, X) 83.693929, var(X) 65617.2326, theta 1.27548703e-03, least squares slope 1.27548703e-03
  overall mean raw 1.050908, adjusted 1.050908


## A2. Adjusted vs unadjusted, per comparison

For each comparison, the difference in means and its unpooled standard error, SE = sqrt(s_t^2 / n_t + s_c^2 / n_c), computed the same way on the raw and the adjusted outcome. Both use the sample variance (ddof = 1), so the only thing that differs between the two columns is the CUPED adjustment. For conversion this SE matches the readout's p(1 - p) / n version to within a factor of sqrt(n / (n - 1)), which is negligible at this size.

Intervals use the normal approximation, difference +/- z x SE, at the same level as the readout.
- conversion, 98.33% for all three comparisons
- spend, the Holm-matched level from `02_primary_readout.ipynb` section 10 (98.33%, 97.5%, 95% by rank)

The unadjusted interval here is the normal (Welch-style) one, not the BCa interval from the readout, so the comparison is like for like.

Then the efficiency numbers.
- variance reduction = 1 - (SE_adj / SE_raw)^2
- equivalent n = n / (1 - variance reduction), using the smaller arm of the comparison
- customers saved = equivalent n - n

In [3]:
spend_levels = pd.read_csv(TABLES / "spend_rank_matched.csv").set_index("comparison")["holm_bar"]
arms = {arm: g for arm, g in df.groupby("segment")}

def diff_se(col, arm_t, arm_c):
    y_t, y_c = arms[arm_t][col].to_numpy(dtype=float), arms[arm_c][col].to_numpy(dtype=float)
    return y_t.mean() - y_c.mean(), np.sqrt(y_t.var(ddof=1) / len(y_t) + y_c.var(ddof=1) / len(y_c)), len(y_t), len(y_c)

rows = []
for metric in ["conversion", "spend"]:
    for arm_t, arm_c in comparisons:
        comp = f"{arm_t} vs {arm_c}"
        lvl_alpha = ALPHA_ADJ if metric == "conversion" else spend_levels[comp]
        z_crit = stats.norm.ppf(1 - lvl_alpha / 2)
        d_raw, se_raw, n_t, n_c = diff_se(metric, arm_t, arm_c)
        d_adj, se_adj, _, _ = diff_se(f"{metric}_cuped", arm_t, arm_c)
        reduction = 1 - (se_adj / se_raw) ** 2
        n_small = min(n_t, n_c)
        rows.append({
            "metric": metric, "comparison": comp, "level": 1 - lvl_alpha,
            "diff_raw": d_raw, "se_raw": se_raw, "lo_raw": d_raw - z_crit * se_raw, "hi_raw": d_raw + z_crit * se_raw,
            "diff_cuped": d_adj, "se_cuped": se_adj, "lo_cuped": d_adj - z_crit * se_adj, "hi_cuped": d_adj + z_crit * se_adj,
            "z_cuped": d_adj / se_adj, "p_cuped": 2 * stats.norm.sf(abs(d_adj / se_adj)),
            "variance_reduction": reduction, "n_smaller_arm": n_small,
            "equivalent_n": n_small / (1 - reduction), "customers_saved": n_small / (1 - reduction) - n_small,
        })
cuped = pd.DataFrame(rows)
cuped[["metric", "comparison", "level", "diff_raw", "se_raw", "diff_cuped", "se_cuped"]].style.format(
    {"level": "{:.2%}", "diff_raw": "{:.6f}", "se_raw": "{:.6f}", "diff_cuped": "{:.6f}", "se_cuped": "{:.6f}"})

,metric,comparison,level,diff_raw,se_raw,diff_cuped,se_cuped
0,conversion,Mens E-Mail vs No E-Mail,98.33%,0.006805,0.000921,0.006784,0.000920
1,conversion,Womens E-Mail vs No E-Mail,98.33%,0.003111,0.000823,0.003093,0.000822
2,conversion,Mens E-Mail vs Womens E-Mail,98.33%,0.003694,0.000995,0.003691,0.000995
3,spend,Mens E-Mail vs No E-Mail,98.33%,0.769827,0.145247,0.767336,0.145215
4,spend,Womens E-Mail vs No E-Mail,97.50%,0.424412,0.130333,0.422303,0.130310
5,spend,Mens E-Mail vs Womens E-Mail,95.00%,0.345415,0.159617,0.345033,0.159572


In [4]:
cuped[["metric", "comparison", "lo_raw", "hi_raw", "lo_cuped", "hi_cuped", "p_cuped"]].style.format(
    {"lo_raw": "{:.6f}", "hi_raw": "{:.6f}", "lo_cuped": "{:.6f}", "hi_cuped": "{:.6f}", "p_cuped": "{:.4e}"})

,metric,comparison,lo_raw,hi_raw,lo_cuped,hi_cuped,p_cuped
0,conversion,Mens E-Mail vs No E-Mail,0.004600,0.009010,0.004580,0.008987,1.7070e-13
1,conversion,Womens E-Mail vs No E-Mail,0.001142,0.005081,0.001124,0.005062,1.6925e-04
2,conversion,Mens E-Mail vs Womens E-Mail,0.001312,0.006076,0.001309,0.006072,2.0686e-04
3,spend,Mens E-Mail vs No E-Mail,0.422110,1.117544,0.419694,1.114978,1.2630e-07
4,spend,Womens E-Mail vs No E-Mail,0.132284,0.716541,0.130226,0.714379,1.1921e-03
5,spend,Mens E-Mail vs Womens E-Mail,0.032571,0.658259,0.032279,0.657788,3.0599e-02


In [5]:
cuped[["metric", "comparison", "variance_reduction", "n_smaller_arm", "equivalent_n", "customers_saved"]].style.format(
    {"variance_reduction": "{:.4%}", "equivalent_n": "{:.2f}", "customers_saved": "{:.2f}"})

,metric,comparison,variance_reduction,n_smaller_arm,equivalent_n,customers_saved
0,conversion,Mens E-Mail vs No E-Mail,0.0881%,21306,21324.78,18.78
1,conversion,Womens E-Mail vs No E-Mail,0.0633%,21306,21319.49,13.49
2,conversion,Mens E-Mail vs Womens E-Mail,0.0998%,21307,21328.29,21.29
3,spend,Mens E-Mail vs No E-Mail,0.0433%,21306,21315.23,9.23
4,spend,Womens E-Mail vs No E-Mail,0.0357%,21306,21313.60,7.60
5,spend,Mens E-Mail vs Womens E-Mail,0.0574%,21307,21319.25,12.25


## A3. Did CUPED do what the pooled check predicted?

`00_cuped_covariate_check.ipynb` predicted a variance reduction of about R^2 for history alone, from pooled data with no arm split. Comparing that prediction with the reductions measured above.

In [6]:
pred = pd.read_csv(TABLES / "cuped_covariate_check.csv")
pred = pred[pred["covariates"] == "history only"].set_index("outcome")["adj_r2"]
for metric in ["conversion", "spend"]:
    got = cuped.loc[cuped["metric"] == metric, "variance_reduction"]
    print(f"{metric}: predicted (pooled adjusted R^2) {pred[metric]:.4%}, measured per comparison {', '.join(f'{v:.4%}' for v in got)}")

conversion: predicted (pooled adjusted R^2) 0.0849%, measured per comparison 0.0881%, 0.0633%, 0.0998%
spend: predicted (pooled adjusted R^2) 0.0457%, measured per comparison 0.0433%, 0.0357%, 0.0574%


# Part B. Pre-registered segments

## B1. Levels and arm counts

Same segment definitions as `00_segment_family_power.ipynb`, recency in tertiles (3 or less, 4 to 8, 9 or more months). Inside each level, conversion counts by arm. Before any test runs, every arm in every level must have at least one conversion, otherwise the unpooled SE for that cell would be zero and the z-test would be wrong without warning.

In [7]:
df["recency_bin"] = pd.cut(df["recency"], bins=[0, 3, 8, 12], labels=["<=3", "4-8", "9+"], include_lowest=True)
df["newbie_s"] = df["newbie"].astype(str)
segment_vars = {"newbie": "newbie_s", "channel": "channel", "zip_code": "zip_code", "recency": "recency_bin"}

counts = []
for name, col in segment_vars.items():
    g = df.groupby([col, "segment"], observed=True)["conversion"].agg(n="count", conversions="sum").reset_index()
    g = g.rename(columns={col: "level"})
    g["variable"] = name
    g["level"] = g["level"].astype(str)
    counts.append(g)
counts = pd.concat(counts)[["variable", "level", "segment", "n", "conversions"]]
n_levels = counts[["variable", "level"]].drop_duplicates().shape[0]
assert n_levels == 11, n_levels
assert (counts["conversions"] > 0).all(), counts[counts["conversions"] == 0]
print(f"levels: {n_levels}, smallest conversion count in any arm and level: {counts['conversions'].min()}")
counts.pivot_table(index=["variable", "level"], columns="segment", values=["n", "conversions"])

levels: 11, smallest conversion count in any arm and level: 18


conversions                                   n  \
segment               Mens E-Mail No E-Mail Womens E-Mail Mens E-Mail   
variable level                                                          
channel  Multichannel        44.0      18.0          36.0      2577.0   
         Phone              100.0      50.0          67.0      9240.0   
         Web                123.0      54.0          86.0      9490.0   
newbie   0                  148.0      82.0          92.0     10621.0   
         1                  119.0      40.0          97.0     10686.0   
recency  4-8                 75.0      37.0          52.0      7157.0   
         9+                  70.0      18.0          49.0      6681.0   
         <=3                122.0      67.0          88.0      7469.0   
zip_code Rural               44.0      26.0          37.0      3243.0   
         Surburban          118.0      48.0          73.0      9501.0   
         Urban              105.0      48.0          79.0      8563.0   

                                               
segment               No E-Mail Womens E-Mail  
variable level                                 
channel  Multichannel    2606.0        2579.0  
         Phone           9327.0        9454.0  
         Web             9373.0        9354.0  
newbie   0              10611.0       10624.0  
         1              10695.0       10763.0  
recency  4-8             7339.0        7269.0  
         9+              6529.0        6632.0  
         <=3             7438.0        7486.0  
zip_code Rural           3139.0        3181.0  
         Surburban       9625.0        9650.0  
         Urban           8542.0        8556.0

## B2. One test worked by hand

Same z-test as the primary readout, for newbie = 0, Mens E-Mail vs No E-Mail.

rate = conversions / n in each arm, difference = rate_t - rate_c

SE = sqrt( rate_t (1 - rate_t) / n_t + rate_c (1 - rate_c) / n_c ), unpooled

z = difference / SE, two-sided p = 2 x P(Z > |z|)

Two-sided matters here, the harm flag needs a test that can see a negative lift.

In [8]:
cell = counts.set_index(["variable", "level", "segment"])

def seg_test(var, lvl, arm_t, arm_c):
    t, c = cell.loc[(var, lvl, arm_t)], cell.loc[(var, lvl, arm_c)]
    r_t, r_c = t["conversions"] / t["n"], c["conversions"] / c["n"]
    se = np.sqrt(r_t * (1 - r_t) / t["n"] + r_c * (1 - r_c) / c["n"])
    z = (r_t - r_c) / se
    return {"n_t": int(t["n"]), "conv_t": int(t["conversions"]), "n_c": int(c["n"]), "conv_c": int(c["conversions"]),
            "rate_t": r_t, "rate_c": r_c, "diff": r_t - r_c, "se": se, "z": z, "p_raw": 2 * stats.norm.sf(abs(z))}

ex = seg_test("newbie", "0", "Mens E-Mail", "No E-Mail")
for k, v in ex.items():
    print(f"{k}: {v:.6f}" if isinstance(v, float) else f"{k}: {v}")

n_t: 10621
conv_t: 148
n_c: 10611
conv_c: 82
rate_t: 0.013935
rate_c: 0.007728
diff: 0.006207
se: 0.001420
z: 4.371043
p_raw: 0.000012


## B3. All 22 tests

In [9]:
levels = counts[["variable", "level"]].drop_duplicates().itertuples(index=False)
seg = pd.DataFrame([
    {"variable": v, "level": l, "comparison": f"{arm_t} vs No E-Mail", **seg_test(v, l, arm_t, "No E-Mail")}
    for v, l in levels for arm_t in ["Mens E-Mail", "Womens E-Mail"]
])
M = len(seg)
assert M == 22, M
seg[["variable", "level", "comparison", "n_t", "conv_t", "n_c", "conv_c", "rate_t", "rate_c", "diff", "se", "z", "p_raw"]].style.format(
    {"rate_t": "{:.4%}", "rate_c": "{:.4%}", "diff": "{:.4%}", "se": "{:.4%}", "z": "{:.4f}", "p_raw": "{:.4e}"})

,variable,level,comparison,n_t,conv_t,n_c,conv_c,rate_t,rate_c,diff,se,z,p_raw
0,newbie,0,Mens E-Mail vs No E-Mail,10621,148,10611,82,1.3935%,0.7728%,0.6207%,0.1420%,4.3710,1.2365e-05
1,newbie,0,Womens E-Mail vs No E-Mail,10624,92,10611,82,0.8660%,0.7728%,0.0932%,0.1237%,0.7532,4.5136e-01
2,newbie,1,Mens E-Mail vs No E-Mail,10686,119,10695,40,1.1136%,0.3740%,0.7396%,0.1174%,6.2984,3.0075e-10
3,newbie,1,Womens E-Mail vs No E-Mail,10763,97,10695,40,0.9012%,0.3740%,0.5272%,0.1085%,4.8573,1.1902e-06
4,channel,Multichannel,Mens E-Mail vs No E-Mail,2577,44,2606,18,1.7074%,0.6907%,1.0167%,0.3024%,3.3621,7.7355e-04
5,channel,Multichannel,Womens E-Mail vs No E-Mail,2579,36,2606,18,1.3959%,0.6907%,0.7052%,0.2823%,2.4980,1.2490e-02
6,channel,Phone,Mens E-Mail vs No E-Mail,9240,100,9327,50,1.0823%,0.5361%,0.5462%,0.1315%,4.1522,3.2936e-05
7,channel,Phone,Womens E-Mail vs No E-Mail,9454,67,9327,50,0.7087%,0.5361%,0.1726%,0.1147%,1.5047,1.3240e-01
8,channel,Web,Mens E-Mail vs No E-Mail,9490,123,9373,54,1.2961%,0.5761%,0.7200%,0.1400%,5.1438,2.6926e-07
9,channel,Web,Womens E-Mail vs No E-Mail,9354,86,9373,54,0.9194%,0.5761%,0.3433%,0.1259%,2.7266,6.3986e-03


## B4. Benjamini-Hochberg, by hand

With m = 22 tests and q = 0.05.

1. Sort the raw p-values from smallest to largest, rank k = 1 to m
2. Each one's bar is (k / m) x q
3. Find the largest k whose p-value is at or below its bar. That test and every test ranked before it are significant. R is how many that is
4. Adjusted p for rank k = p_(k) x m / k, then take the running minimum from the largest rank down, capped at 1

Cross-checked against statsmodels.

In [10]:
Q = ALPHA
bh = seg[["variable", "level", "comparison", "p_raw"]].sort_values("p_raw").reset_index()
bh["rank"] = np.arange(1, M + 1)
bh["bar"] = bh["rank"] / M * Q
bh["p_at_or_below_bar"] = bh["p_raw"] <= bh["bar"]
R = int(bh.loc[bh["p_at_or_below_bar"], "rank"].max()) if bh["p_at_or_below_bar"].any() else 0
bh["bh_sig"] = bh["rank"] <= R
bh["p_bh"] = (bh["p_raw"] * M / bh["rank"])[::-1].cummin()[::-1].clip(upper=1)

_, p_bh_sm, _, _ = multipletests(bh["p_raw"], alpha=Q, method="fdr_bh")
bh["p_bh_statsmodels"] = p_bh_sm
assert np.allclose(bh["p_bh"], bh["p_bh_statsmodels"])
assert (bh["bh_sig"] == (bh["p_bh"] <= Q)).all()
print(f"R, number of BH-significant tests: {R} of {M}")
bh.style.format({"p_raw": "{:.4e}", "bar": "{:.5f}", "p_bh": "{:.4e}", "p_bh_statsmodels": "{:.4e}"})

R, number of BH-significant tests: 16 of 22


,index,variable,level,comparison,p_raw,rank,bar,p_at_or_below_bar,bh_sig,p_bh,p_bh_statsmodels
0,2,newbie,1,Mens E-Mail vs No E-Mail,3.0075e-10,1,0.00227,True,True,6.6166e-09,6.6166e-09
1,12,zip_code,Surburban,Mens E-Mail vs No E-Mail,3.2017e-08,2,0.00455,True,True,2.8394e-07,2.8394e-07
2,20,recency,9+,Mens E-Mail vs No E-Mail,3.8719e-08,3,0.00682,True,True,2.8394e-07,2.8394e-07
3,8,channel,Web,Mens E-Mail vs No E-Mail,2.6926e-07,4,0.00909,True,True,1.4809e-06,1.4809e-06
4,3,newbie,1,Womens E-Mail vs No E-Mail,1.1902e-06,5,0.01136,True,True,5.2370e-06,5.2370e-06
5,14,zip_code,Urban,Mens E-Mail vs No E-Mail,3.8627e-06,6,0.01364,True,True,1.4163e-05,1.4163e-05
6,0,newbie,0,Mens E-Mail vs No E-Mail,1.2365e-05,7,0.01591,True,True,3.8863e-05,3.8863e-05
7,6,channel,Phone,Mens E-Mail vs No E-Mail,3.2936e-05,8,0.01818,True,True,9.0574e-05,9.0574e-05
8,16,recency,<=3,Mens E-Mail vs No E-Mail,6.2803e-05,9,0.02045,True,True,1.5352e-04,1.5352e-04
9,21,recency,9+,Womens E-Mail vs No E-Mail,1.7817e-04,10,0.02273,True,True,3.9198e-04,3.9198e-04


## B5. FCR intervals matched to BH, with Newcombe beside

Interval level = 1 - R q / m (Benjamini and Yekutieli 2005). If R = 0, intervals are shown at 95% and labeled unadjusted.

- Wald interval, difference +/- z x SE at that level, plus the plain 95% Wald interval as context
- Newcombe hybrid score interval at the same level, from statsmodels as in the primary readout

Two checks follow. Duality, a Wald interval at the FCR level excludes zero exactly when BH calls the test significant. Then the two flags.
- sensitive to interval method, Wald and Newcombe disagree on whether zero is excluded
- harm, BH-significant with a negative lift

In [11]:
seg = seg.merge(bh[["variable", "level", "comparison", "rank", "p_bh", "bh_sig"]], on=["variable", "level", "comparison"])
alpha_fcr = R * Q / M if R > 0 else ALPHA
fcr_label = f"{1 - alpha_fcr:.4%} FCR" if R > 0 else "95% unadjusted (R = 0)"
z_fcr, z_95 = stats.norm.ppf(1 - alpha_fcr / 2), stats.norm.ppf(1 - ALPHA / 2)
print(f"alpha_fcr = R q / m = {R} x {Q} / {M} = {alpha_fcr:.6f}, interval level {fcr_label}, z {z_fcr:.4f}")

seg["wald_lo_fcr"], seg["wald_hi_fcr"] = seg["diff"] - z_fcr * seg["se"], seg["diff"] + z_fcr * seg["se"]
seg["wald_lo_95"], seg["wald_hi_95"] = seg["diff"] - z_95 * seg["se"], seg["diff"] + z_95 * seg["se"]
newc = [confint_proportions_2indep(r.conv_t, r.n_t, r.conv_c, r.n_c, method="newcomb", compare="diff", alpha=alpha_fcr)
        for r in seg.itertuples()]
seg["newc_lo_fcr"], seg["newc_hi_fcr"] = [n[0] for n in newc], [n[1] for n in newc]

seg["wald_excludes_zero"] = (seg["wald_lo_fcr"] > 0) | (seg["wald_hi_fcr"] < 0)
seg["newc_excludes_zero"] = (seg["newc_lo_fcr"] > 0) | (seg["newc_hi_fcr"] < 0)
if R > 0:
    assert (seg["wald_excludes_zero"] == seg["bh_sig"]).all()
    print("duality holds, every FCR Wald interval excludes zero exactly when BH calls the test significant")
seg["flag_method_sensitive"] = seg["wald_excludes_zero"] != seg["newc_excludes_zero"]
seg["flag_harm"] = seg["bh_sig"] & (seg["diff"] < 0)

pp = {c: (lambda v: f"{100 * v:.4f}") for c in ["diff", "wald_lo_fcr", "wald_hi_fcr", "newc_lo_fcr", "newc_hi_fcr", "wald_lo_95", "wald_hi_95"]}
seg[["variable", "level", "comparison", "diff", "p_bh", "bh_sig", "wald_lo_fcr", "wald_hi_fcr", "newc_lo_fcr", "newc_hi_fcr",
     "wald_lo_95", "wald_hi_95", "flag_method_sensitive", "flag_harm"]].style.format(pp | {"p_bh": "{:.4e}"})

alpha_fcr = R q / m = 16 x 0.05 / 22 = 0.036364, interval level 96.3636% FCR, z 2.0928
duality holds, every FCR Wald interval excludes zero exactly when BH calls the test significant


,variable,level,comparison,diff,p_bh,bh_sig,wald_lo_fcr,wald_hi_fcr,newc_lo_fcr,newc_hi_fcr,wald_lo_95,wald_hi_95,flag_method_sensitive,flag_harm
0,newbie,0,Mens E-Mail vs No E-Mail,0.6207,3.8863e-05,True,0.3235,0.9179,0.3247,0.9243,0.3424,0.8990,False,False
1,newbie,0,Womens E-Mail vs No E-Mail,0.0932,4.5136e-01,False,-0.1657,0.3521,-0.1681,0.3560,-0.1493,0.3357,False,False
2,newbie,1,Mens E-Mail vs No E-Mail,0.7396,6.6166e-09,True,0.4938,0.9854,0.4977,0.9955,0.5094,0.9698,False,False
3,newbie,1,Womens E-Mail vs No E-Mail,0.5272,5.2370e-06,True,0.3001,0.7544,0.3022,0.7634,0.3145,0.7400,False,False
4,channel,Multichannel,Mens E-Mail vs No E-Mail,1.0167,1.4182e-03,True,0.3838,1.6496,0.3872,1.6932,0.4240,1.6094,False,False
5,channel,Multichannel,Womens E-Mail vs No E-Mail,0.7052,1.8318e-02,True,0.1144,1.2960,0.1113,1.3366,0.1519,1.2585,False,False
6,channel,Phone,Mens E-Mail vs No E-Mail,0.5462,9.0574e-05,True,0.2709,0.8215,0.2726,0.8307,0.2884,0.8040,False,False
7,channel,Phone,Womens E-Mail vs No E-Mail,0.1726,1.4563e-01,False,-0.0675,0.4127,-0.0700,0.4188,-0.0522,0.3975,False,False
8,channel,Web,Mens E-Mail vs No E-Mail,0.7200,1.4809e-06,True,0.4270,1.0129,0.4293,1.0219,0.4456,0.9943,False,False
9,channel,Web,Womens E-Mail vs No E-Mail,0.3433,1.0055e-02,True,0.0798,0.6067,0.0795,0.6142,0.0965,0.5900,False,False


Differences and interval bounds in the table above are in percentage points.

## B6. Summary

In [12]:
print(f"BH-significant: {R} of {M} at q = {Q}")
for r in seg[seg["bh_sig"]].sort_values("p_bh").itertuples():
    print(f"  {r.variable} {r.level}, {r.comparison}: {100 * r.diff:+.4f}pp, BH p {r.p_bh:.4e}, "
          f"FCR Wald ({100 * r.wald_lo_fcr:.4f}, {100 * r.wald_hi_fcr:.4f})")
print(f"flagged sensitive to interval method: {int(seg['flag_method_sensitive'].sum())}")
for r in seg[seg["flag_method_sensitive"]].itertuples():
    print(f"  {r.variable} {r.level}, {r.comparison}")
print(f"flagged harm (BH-significant negative lift): {int(seg['flag_harm'].sum())}")
for r in seg[seg["flag_harm"]].itertuples():
    print(f"  {r.variable} {r.level}, {r.comparison}: {100 * r.diff:+.4f}pp")
print(f"negative point estimates (any significance): {int((seg['diff'] < 0).sum())}")

BH-significant: 16 of 22 at q = 0.05
  newbie 1, Mens E-Mail vs No E-Mail: +0.7396pp, BH p 6.6166e-09, FCR Wald (0.4938, 0.9854)
  zip_code Surburban, Mens E-Mail vs No E-Mail: +0.7433pp, BH p 2.8394e-07, FCR Wald (0.4620, 1.0246)
  recency 9+, Mens E-Mail vs No E-Mail: +0.7721pp, BH p 2.8394e-07, FCR Wald (0.4781, 1.0660)
  channel Web, Mens E-Mail vs No E-Mail: +0.7200pp, BH p 1.4809e-06, FCR Wald (0.4270, 1.0129)
  newbie 1, Womens E-Mail vs No E-Mail: +0.5272pp, BH p 5.2370e-06, FCR Wald (0.3001, 0.7544)
  zip_code Urban, Mens E-Mail vs No E-Mail: +0.6643pp, BH p 1.4163e-05, FCR Wald (0.3633, 0.9653)
  newbie 0, Mens E-Mail vs No E-Mail: +0.6207pp, BH p 3.8863e-05, FCR Wald (0.3235, 0.9179)
  channel Phone, Mens E-Mail vs No E-Mail: +0.5462pp, BH p 9.0574e-05, FCR Wald (0.2709, 0.8215)
  recency <=3, Mens E-Mail vs No E-Mail: +0.7326pp, BH p 1.5352e-04, FCR Wald (0.3495, 1.1158)
  recency 9+, Womens E-Mail vs No E-Mail: +0.4631pp, BH p 3.9198e-04, FCR Wald (0.2045, 0.7218)
  recenc

## Save results

In [13]:
cuped.to_csv(TABLES / "cuped_readout.csv", index=False)
seg.assign(interval_level=fcr_label).to_csv(TABLES / "segment_readout.csv", index=False)
bh.drop(columns="index").to_csv(TABLES / "segment_bh.csv", index=False)
print("saved results/tables/cuped_readout.csv, segment_readout.csv, segment_bh.csv")

saved results/tables/cuped_readout.csv, segment_readout.csv, segment_bh.csv
